# 🛒 Web Scraping — Notebook 4
## End-to-End Project: Full E-Commerce Catalogue Scraper

---

## 🎯 Project Goal

Build a **complete production scraper** for `books.toscrape.com` that:

- Scrapes **all 1000 books** across **all 50 pages**
- Extracts: title, price, rating, availability, category, detail URL
- Stores everything in a **SQLite database**
- Handles errors, retries and rate limiting throughout
- Exports a clean **CSV report** at the end
- Generates **data insights** from scraped data

### Why This Project?

This is the type of task you'd get in a real job or internship:
_"We need all products from this catalogue with their prices and ratings — stored in a database."_

### Tools Used
```
requests + BS4    -> Fetch and parse (static site, no JS needed)
SQLite            -> Store structured data
csv               -> Export final report
logging           -> Track scraper progress
time + random     -> Polite rate limiting
```

### Project Architecture

```
books.toscrape.com
       │
       ├── Page 1  -> 20 books
       ├── Page 2  -> 20 books
       │   ...
       └── Page 50 -> 20 books = 1000 books total
                                       │
                               Scrape each book:
                               title, price, rating,
                               availability, category
                                       │
                               ┌───────────────┐
                               │  SQLite DB    │
                               │  books.db     │
                               └───────┬───────┘
                                       │
                               Export → books_report.csv
                               Analyse → insights
```

## ⚙️ Step 1 — Setup & Inspecting the Target

Before writing any scraping code, always inspect the site structure.

### What One Book Card Looks Like (DevTools)

```html
<article class="product_pod">
  <div class="image_container">
    <a href="catalogue/book-name_1/index.html">
      <img src="..." alt="Book Title" class="thumbnail">
    </a>
  </div>
  <p class="star-rating Three"></p>              <- Rating in class!
  <h3>
    <a href="catalogue/book-name_1/index.html"
       title="A Light in the Attic">A Light ...</a>  <- Full title in 'title'
  </h3>
  <div class="product_price">
    <p class="price_color">£51.77</p>
    <p class="availability">In stock</p>
  </div>
</article>
```

### Pagination Structure
```
Page 1: https://books.toscrape.com/catalogue/page-1.html
Page 2: https://books.toscrape.com/catalogue/page-2.html
....
Page 50: https://books.toscrape.com/catalogue/page-50.html

Next button: <li class="next"><a href="page-2.html">next</a></li>
```

In [ ]:
# STEP 1 — Imports, Config, Logging Setup

import requests
import re
import csv
import json
import os
import time
import random
import logging
import sqlite3
from datetime import datetime
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup

# tqdm: pip install tqdm (progress bar for long scrapes)
try:
    from tqdm import tqdm
    TQDM_AVAILABLE = True
except ImportError:
    TQDM_AVAILABLE = False
    print('tqdm not installed. Run: pip install tqdm')
    print('Falling back to log-based progress.')

# ── Config ──
BASE_URL        = 'https://books.toscrape.com/catalogue/'
START_URL       = 'https://books.toscrape.com/catalogue/page-1.html'
DB_FILE         = 'books_catalogue.db'
CSV_FILE        = 'books_report.csv'
CHECKPOINT_FILE = 'books_checkpoint.json'
DELAY_MIN       = 0.8
DELAY_MAX       = 1.5
MAX_RETRIES     = 3

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
    )
}

RATING_MAP = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

# ── Logging ──
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[
        logging.FileHandler('books_scraper.log', mode='w'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('books_scraper')
log.info('Books scraper initialised')

# ── robots.txt check ──
def can_scrape(base_url, target_url):
    rp = RobotFileParser()
    rp.set_url(base_url.rstrip('/') + '/robots.txt')
    try:
        rp.read()
        return rp.can_fetch('*', target_url)
    except Exception:
        return True

allowed = can_scrape('https://books.toscrape.com', START_URL)
print(f'robots.txt check: {"✅ ALLOWED" if allowed else "❌ BLOCKED"}')
print('Setup complete!')


## 🗄️ Step 2 — Database Schema & Helper Functions

Define the database structure and all helper functions before writing the main scraper.
This is good practice — keeps the scraper logic clean and readable.

### DB Schema
```sql
CREATE TABLE books (
    id           INTEGER PRIMARY KEY,
    title        TEXT UNIQUE,       <- UNIQUE prevents duplicates
    price        REAL,
    rating       INTEGER,           <- 1-5 (converted from word)
    availability TEXT,
    detail_url   TEXT,
    page_num     INTEGER,
    scraped_at   DATETIME
)
```

In [ ]:
# STEP 2 — Database + Helper Functions (with Session & context manager)

# ── Session factory ──
def make_session():
    """Create a configured Session. Reuse it across ALL page fetches."""
    s = requests.Session()
    s.headers.update(HEADERS)
    return s


# ── Database (context manager pattern) ──
def init_db():
    with sqlite3.connect(DB_FILE) as conn:   # 'with' auto-closes + commits
        conn.execute('''
            CREATE TABLE IF NOT EXISTS books (
                id           INTEGER PRIMARY KEY AUTOINCREMENT,
                title        TEXT UNIQUE,
                price        REAL,
                rating       INTEGER,
                availability TEXT,
                detail_url   TEXT,
                page_num     INTEGER,
                scraped_at   TEXT
            )
        ''')
    log.info(f'Database ready: {DB_FILE}')


def save_books(books):
    saved = skipped = 0
    with sqlite3.connect(DB_FILE) as conn:   # auto-closes!
        for b in books:
            c = conn.execute(
                'INSERT OR IGNORE INTO books '
                '(title,price,rating,availability,detail_url,page_num,scraped_at)'
                ' VALUES (?,?,?,?,?,?,?)',
                (b['title'], b['price'], b['rating'],
                 b['availability'], b['detail_url'],
                 b['page_num'], b['scraped_at'])
            )
            if c.rowcount > 0: saved += 1
            else: skipped += 1
    return saved, skipped


# ── Validation ──
def validate_book(book):
    """Returns list of error strings. Empty list = valid."""
    errors = []
    if not book.get('title') or book['title'] == 'N/A':
        errors.append('Missing title')
    if book.get('price', 0) <= 0:
        errors.append(f'Invalid price: {book.get("price")}')
    if book.get('rating', -1) not in range(0, 6):
        errors.append(f'Invalid rating: {book.get("rating")}')
    return errors


# ── Helper: Safe extraction ──
def safe_text(tag, default='N/A'):
    return tag.get_text(strip=True) if tag else default

def to_float(text, default=0.0):
    cleaned = re.sub(r'[^\d.]', '', text or '')
    return float(cleaned) if cleaned else default


# ── Fetch with retry using Session ──
def fetch(session, url):
    """Fetch URL using the shared session. Retries on failure."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, timeout=10)
            r.raise_for_status()
            return BeautifulSoup(r.text, 'html.parser')
        except requests.exceptions.HTTPError as e:
            log.error(f'HTTP {e.response.status_code} [{attempt}]: {url}')
            if e.response.status_code == 404: return None
        except requests.exceptions.RequestException as e:
            log.warning(f'Attempt {attempt} failed: {e}')
            if attempt < MAX_RETRIES:
                time.sleep(2 ** (attempt - 1))
    return None


# ── Checkpoint helpers ──
def save_checkpoint(page_num, next_url, total_saved):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'last_page': page_num, 'next_url': next_url,
                   'total_saved': total_saved,
                   'saved_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')}, f, indent=2)

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            cp = json.load(f)
        log.info(f'Resuming from checkpoint: page {cp["last_page"]+1}')
        return cp
    return None

def clear_checkpoint():
    if os.path.exists(CHECKPOINT_FILE): os.remove(CHECKPOINT_FILE)


# ── Test ──
init_db()
session = make_session()
test_soup = fetch(session, START_URL)
if test_soup:
    cards = test_soup.find_all('article', class_='product_pod')
    log.info(f'Test fetch OK — {len(cards)} books on page 1')
    print(f'DB, Session, and helpers ready! Found {len(cards)} books on page 1.')
else:
    log.error('Test fetch failed!')


## 🔍 Step 3 — Parse a Single Page

Build and test the parser on one page before running it on all 50.
This is best practice — verify the extraction is correct before the full run.

### Extraction Logic Per Book Card

```python
for card in soup.find_all('article', class_='product_pod'):

    # Title: inside <a title='...'> of <h3>
    h3    = card.find('h3')
    title = h3.find('a')['title']    if h3 and h3.find('a') else 'N/A'

    # Price: <p class='price_color'> e.g. '£51.77'
    price = to_float(safe_text(card.find('p', class_='price_color')))

    # Rating: second class of <p class='star-rating Three'>
    r_tag  = card.find('p', class_='star-rating')
    rating = RATING_MAP.get(r_tag['class'][1], 0)  if r_tag else 0

    # Availability: <p class='availability'>
    avail = safe_text(card.find('p', class_='availability'))

    # Detail URL: href of <a> inside <h3>
    href  = h3.find('a')['href']     if h3 and h3.find('a') else ''
    url   = urljoin(BASE_URL, href)
```

In [ ]:
# STEP 3 — Parse One Page (with validation)

def parse_page(soup, page_num):
    """Extract all books from a single page soup object."""
    cards      = soup.find_all('article', class_='product_pod')
    books      = []
    invalid    = []
    now        = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    for card in cards:
        h3    = card.find('h3')
        a_tag = h3.find('a') if h3 else None

        title  = a_tag['title']  if a_tag and a_tag.get('title') else safe_text(a_tag)
        href   = a_tag['href']   if a_tag and a_tag.get('href')  else ''

        price_tag  = card.find('p', class_='price_color')
        rating_tag = card.find('p', class_='star-rating')
        avail_tag  = card.find('p', class_='availability')

        price      = to_float(safe_text(price_tag))
        rating_str = rating_tag['class'][1] if rating_tag and len(rating_tag.get('class', [])) > 1 else 'Zero'
        rating     = RATING_MAP.get(rating_str, 0)
        avail      = safe_text(avail_tag)
        detail_url = urljoin(BASE_URL, href)

        book = {
            'title'       : title,
            'price'       : price,
            'rating'      : rating,
            'availability': avail,
            'detail_url'  : detail_url,
            'page_num'    : page_num,
            'scraped_at'  : now
        }

        # Validate before accepting
        errors = validate_book(book)
        if errors:
            log.warning(f'Invalid item on page {page_num}: {errors}')
            invalid.append({'book': book, 'errors': errors})
        else:
            books.append(book)

    if invalid:
        log.warning(f'Page {page_num}: {len(invalid)} invalid items skipped')

    return books, invalid


# ── Test on page 1 ──
books_p1, invalid_p1 = parse_page(test_soup, page_num=1)

print(f'Parsed {len(books_p1)} valid books, {len(invalid_p1)} invalid from page 1')
print()
print(f'{"Title":<45} {"Price":>7} {"Rating":>7} {"Stock"}')
print('-' * 75)
for b in books_p1[:5]:
    print(f'{b["title"][:44]:<45} £{b["price"]:>5.2f} {b["rating"]:>6}⭐  {b["availability"]}')
print('... and more')


## 🔄 Step 4 — Pagination: Scrape All 50 Pages

Now that our single-page parser works, wrap it in a pagination loop.

### Finding the Next Page

```html
<!-- At the bottom of each page -->
<li class="next">
  <a href="page-2.html">next</a>
</li>
```

```python
next_li  = soup.find('li', class_='next')
next_url = urljoin(current_url, next_li.find('a')['href']) if next_li else None
# next_url = None on the last page -> loop stops
```

### Progress Tracking
With 50 pages, you want to see progress:
```
Page  1/50 | 20 books | Total:   20 | Saved:   20
Page  2/50 | 20 books | Total:   40 | Saved:   40
...
Page 50/50 | 20 books | Total: 1000 | Saved: 1000
```

In [ ]:
# STEP 4 — Paginated Scraper (Session + tqdm progress bar + Checkpoint)

def scrape_all_books(start_url=START_URL, max_pages=None):
    """
    Scrape all books using:
    - Shared Session (faster)
    - tqdm progress bar
    - Checkpoint/Resume (crash-safe)
    - Validation before save
    """
    # ── Check for checkpoint (resume if crash happened) ──
    cp       = load_checkpoint()
    page_num = cp['last_page'] + 1  if cp else 1
    current_url = cp['next_url']    if cp else start_url
    total_saved = cp['total_saved'] if cp else 0
    total_found = 0
    total_invalid = 0

    if cp:
        print(f'Resuming from page {page_num} ({total_saved} already saved)')
    else:
        print('Fresh start — no checkpoint found')

    log.info(f'Starting scrape from page {page_num}: {current_url}')

    # ── tqdm setup ──
    # 50 pages total; start from where we left off
    total_pages  = max_pages or 50
    pages_done   = page_num - 1
    pages_left   = total_pages - pages_done

    progress = tqdm(
        total=total_pages,
        initial=pages_done,
        desc='Scraping',
        unit='page',
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} pages [{elapsed}<{remaining}]'
    ) if TQDM_AVAILABLE else None

    # ── Shared Session ──
    session = make_session()

    try:
        while current_url:
            if max_pages and page_num > max_pages:
                log.info(f'max_pages={max_pages} reached. Stopping.')
                break

            soup = fetch(session, current_url)
            if not soup:
                log.error(f'Failed page {page_num}. Saving checkpoint and stopping.')
                save_checkpoint(page_num - 1, current_url, total_saved)
                break

            # Parse + validate
            books, invalid = parse_page(soup, page_num)
            total_found   += len(books)
            total_invalid += len(invalid)

            # Save
            saved, skipped = save_books(books)
            total_saved   += saved

            # Update progress bar
            if progress:
                progress.set_postfix(saved=total_saved, invalid=total_invalid)
                progress.update(1)
            else:
                log.info(f'Page {page_num:2d} | valid={len(books)} invalid={len(invalid)} | '
                         f'saved={saved} | total={total_saved}')

            # Next page
            next_li = soup.find('li', class_='next')
            if next_li and next_li.find('a'):
                next_url    = urljoin(current_url, next_li.find('a')['href'])
                save_checkpoint(page_num, next_url, total_saved)  # checkpoint!
                current_url = next_url
                page_num   += 1
            else:
                current_url = None

            if current_url:
                time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    finally:
        session.close()   # Always close session
        if progress: progress.close()

    clear_checkpoint()    # Success — remove checkpoint file
    log.info(f'Scrape complete! found={total_found} saved={total_saved} invalid={total_invalid}')
    return total_found, total_saved, total_invalid


# ── Run (first 5 pages as demo) ──
print('Scraping first 5 pages as a demo (tqdm progress bar below):')
print('Change max_pages=None to scrape all 50 pages')
print()

found, saved, invalid = scrape_all_books(max_pages=5)

print()
print(f'Books found  : {found}')
print(f'Books saved  : {saved}')
print(f'Invalid items: {invalid}')


## 📊 Step 5 — Query the Database & Generate Insights

Now that data is in SQLite, we can ask interesting questions using SQL.
This is the power of storing in a database instead of a flat CSV.

### Questions We'll Answer

```sql
-- What is the average price?
SELECT AVG(price) FROM books;

-- How many books per rating?
SELECT rating, COUNT(*) FROM books GROUP BY rating ORDER BY rating;

-- Top 5 most expensive books
SELECT title, price FROM books ORDER BY price DESC LIMIT 5;

-- Cheapest 5-star books
SELECT title, price FROM books WHERE rating=5 ORDER BY price LIMIT 5;
```

In [ ]:
# STEP 5 — Query + Insights

conn = sqlite3.connect(DB_FILE)
conn.row_factory = sqlite3.Row

# ── Basic stats ──
stats = conn.execute('''
    SELECT COUNT(*) total,
           ROUND(AVG(price), 2) avg_price,
           ROUND(MIN(price), 2) min_price,
           ROUND(MAX(price), 2) max_price,
           ROUND(AVG(rating), 2) avg_rating
    FROM books
''').fetchone()

print('=' * 50)
print('📊 DATABASE INSIGHTS')
print('=' * 50)
print(f'Total books scraped : {stats["total"]}')
print(f'Average price       : £{stats["avg_price"]}')
print(f'Price range         : £{stats["min_price"]} — £{stats["max_price"]}')
print(f'Average rating      : {stats["avg_rating"]} / 5')
print()

# ── Distribution by rating ──
print('Books by rating:')
stars = '⭐'
for row in conn.execute('SELECT rating, COUNT(*) c FROM books GROUP BY rating ORDER BY rating DESC').fetchall():
    bar = '█' * row['c']
    print(f'  {stars * row["rating"]:10s} ({row["rating"]}⭐): {row["c"]:4d} books  {bar[:40]}')
print()

# ── Top 5 most expensive ──
print('Top 5 most expensive books:')
for row in conn.execute('SELECT title, price FROM books ORDER BY price DESC LIMIT 5').fetchall():
    print(f'  £{row["price"]:5.2f}  {row["title"][:50]}')
print()

# ── Best value: 5 stars, cheapest price ──
print('Best value (5⭐, cheapest price):')
for row in conn.execute(
    'SELECT title, price FROM books WHERE rating=5 ORDER BY price ASC LIMIT 5'
).fetchall():
    print(f'  £{row["price"]:5.2f}  {row["title"][:50]}')

conn.close()

## 💾 Step 6 — Export to CSV

The final step in most scraping pipelines — export the data for stakeholders who prefer Excel/Sheets over databases.

```python
# Pattern: query DB -> write to CSV
conn = sqlite3.connect(DB_FILE)
rows = conn.execute('SELECT * FROM books ORDER BY price').fetchall()

with open('report.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Title', 'Price', 'Rating', ...])  # Header
    writer.writerows(rows)
```

In [ ]:
# STEP 6 — Export to CSV

def export_csv(db_file, csv_file, query=None, order='price ASC'):
    """Export books from DB to CSV with optional filter query."""
    conn   = sqlite3.connect(db_file)
    conn.row_factory = sqlite3.Row

    sql = query or f'SELECT title, price, rating, availability, detail_url, page_num, scraped_at FROM books ORDER BY {order}'
    rows = conn.execute(sql).fetchall()
    conn.close()

    if not rows:
        print('No data to export.')
        return

    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows([dict(r) for r in rows])

    print(f'Exported {len(rows)} rows to {csv_file}')
    return len(rows)


# ── Export all books sorted by price ──
count = export_csv(DB_FILE, CSV_FILE)
print()

# ── Export only 5-star books as a separate file ──
count5 = export_csv(
    DB_FILE,
    'top_rated_books.csv',
    query='SELECT title, price, rating, availability FROM books WHERE rating=5 ORDER BY price'
)
print()

# Preview the CSV
print('CSV preview (first 5 rows):')
print('-' * 70)
with open(CSV_FILE, encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(line.strip()[:100])
        if i >= 5: break

## 🚀 Step 7 — Run the Full Scrape (All 1000 Books)

Everything is built and tested. Now run the complete scrape.

```
Estimated time: ~3-4 minutes
(50 pages × ~3 seconds per page including delay)
```

### What to Expect

```
09:00:01 | INFO     | Starting full scrape
09:00:02 | INFO     | Page  1 | 20 books | Total:   20 | Saved:   20
09:00:04 | INFO     | Page  2 | 20 books | Total:   40 | Saved:   40
...
09:03:45 | INFO     | Page 50 | 20 books | Total: 1000 | Saved: 1000
09:03:45 | INFO     | Scrape complete!
```

> ⚠️ **If you already ran with max_pages=5**, the DB has 100 books.
> Running this cell will add the remaining 900. Duplicates are safely skipped.

In [ ]:
# STEP 7 — Full Scrape (all 50 pages | Session + tqdm + Checkpoint)

print('Starting FULL scrape of all 50 pages...')
print('  ✅ Using shared Session  (faster TCP reuse)')
print('  ✅ tqdm progress bar     (visual progress)')
print('  ✅ Checkpoint on each page (crash-safe resume)')
print('  ✅ Validation before save')
print()

start_time = time.time()

# max_pages=None -> scrapes ALL pages
found, saved, invalid = scrape_all_books(max_pages=None)

elapsed = time.time() - start_time
print()
print(f'✅ DONE!')
print(f'   Books found  : {found}')
print(f'   Books saved  : {saved}')
print(f'   Invalid items: {invalid}')
print(f'   Time taken   : {elapsed:.1f}s ({elapsed/60:.1f} min)')
print()

# Re-export full CSV
export_csv(DB_FILE, CSV_FILE)

import os
print('\nFiles created:')
for fname in [DB_FILE, CSV_FILE, 'top_rated_books.csv', 'books_scraper.log']:
    if os.path.exists(fname):
        size = os.path.getsize(fname)
        print(f'  {fname:35s} {size:>8,} bytes')


## 📈 Step 8 — Final Analysis on All 1000 Books

With all 1000 books in the database, run final insights.

In [ ]:
# STEP 8 — Final Analysis

conn = sqlite3.connect(DB_FILE)
conn.row_factory = sqlite3.Row

total = conn.execute('SELECT COUNT(*) FROM books').fetchone()[0]

print('=' * 60)
print(f'📚 FINAL ANALYSIS — {total} Books')
print('=' * 60)

# ── Price buckets ──
print('\nPrice distribution:')
buckets = [
    ('Under £20',    'price < 20'),
    ('£20 - £40',   'price BETWEEN 20 AND 40'),
    ('£40 - £60',   'price BETWEEN 40 AND 60'),
    ('Over £60',    'price > 60'),
]
for label, condition in buckets:
    count = conn.execute(f'SELECT COUNT(*) FROM books WHERE {condition}').fetchone()[0]
    pct   = count / total * 100
    bar   = '█' * int(pct / 2)
    print(f'  {label:<15} {count:4d} books  ({pct:5.1f}%)  {bar}')

# ── Best value per rating ──
print('\nCheapest book per rating level:')
for r in range(5, 0, -1):
    row = conn.execute(
        'SELECT title, price FROM books WHERE rating=? ORDER BY price ASC LIMIT 1', (r,)
    ).fetchone()
    if row:
        stars = '⭐' * r
        print(f'  {stars} {r}★  £{row["price"]:5.2f}  {row["title"][:45]}')

# ── Most common price points ──
print('\nTop 5 most common price points:')
for row in conn.execute(
    'SELECT ROUND(price,0) p, COUNT(*) c FROM books GROUP BY p ORDER BY c DESC LIMIT 5'
).fetchall():
    print(f'  £{row["p"]:.0f}  -> {row["c"]} books')

conn.close()

print()
print('Project 6 complete! ✅')
print('Files created: books_catalogue.db, books_report.csv, top_rated_books.csv')

## 🔍 Step 9 — Data Quality Report

After a large scrape, always verify the quality of what was stored.
This tells you if your parser has bugs or if the site has inconsistent data.

In [ ]:
# STEP 9 — Data Quality Report

with sqlite3.connect(DB_FILE) as conn:
    conn.row_factory = sqlite3.Row

    total     = conn.execute('SELECT COUNT(*) FROM books').fetchone()[0]
    no_title  = conn.execute("SELECT COUNT(*) FROM books WHERE title='N/A' OR title=''").fetchone()[0]
    zero_price= conn.execute('SELECT COUNT(*) FROM books WHERE price <= 0').fetchone()[0]
    zero_rating=conn.execute('SELECT COUNT(*) FROM books WHERE rating = 0').fetchone()[0]
    out_stock = conn.execute("SELECT COUNT(*) FROM books WHERE availability NOT LIKE '%stock%'").fetchone()[0]
    duplicates= total - conn.execute('SELECT COUNT(DISTINCT title) FROM books').fetchone()[0]

print('=' * 50)
print('🔍 DATA QUALITY REPORT')
print('=' * 50)

checks = [
    ('Total rows',       total,       total,  'rows scraped'),
    ('Missing titles',   no_title,    0,       'rows with no title'),
    ('Zero/bad prices',  zero_price,  0,       'rows with price <= 0'),
    ('Zero ratings',     zero_rating, 0,       'rows with rating = 0'),
    ('Not in stock',     out_stock,   None,    'rows (informational)'),
    ('Duplicates in DB', duplicates,  0,       'duplicate titles'),
]

all_pass = True
for label, value, expected, note in checks:
    if expected is None:
        status = 'ℹ️ '
    elif value == expected:
        status = '✅'
    else:
        status = '⚠️ '
        all_pass = False
    print(f'  {status} {label:<22}: {value:>5}  ({note})')

print()
if all_pass:
    print('Overall: ✅ Data quality looks good!')
else:
    print('Overall: ⚠️  Some issues found — check the ⚠️ rows above.')
    print('         These may need investigation or re-scraping.')


---
# ✅ Notebook 4 — Complete!

## What You Built

A **real, production-grade web scraper** that:

```
[x] Inspected the target site before writing any code
[x] Configured headers, delays, retry limits in one place
[x] Set up structured logging (file + terminal)
[x] Designed a proper SQLite schema upfront
[x] Built safe extraction helpers (safe_text, to_float)
[x] Tested the parser on one page before full run
[x] Scraped all 50 pages with polite delays
[x] Tracked progress page-by-page
[x] Handled duplicate protection (INSERT OR IGNORE)
[x] Queried the DB for meaningful insights
[x] Exported clean CSV report
[x] Final analysis on all 1000 books
```

## Key Patterns to Remember

```python
# The complete pipeline loop (use this as a template):

while current_url:
    soup   = fetch(current_url)          # 1. Fetch with retry
    books  = parse_page(soup, page_num)  # 2. Parse + clean
    saved  = save_books(books)           # 3. Save with dedup
    log.info(f'Page {page_num}: {saved} saved')

    next_li     = soup.find('li', class_='next')  # 4. Find next
    current_url = urljoin(current_url, ...) if next_li else None
    time.sleep(random.uniform(1, 2))      # 5. Be polite!
```

---

## 🎯 What's in Notebook 5

| Topic | Details |
|-------|---------|
| Multi-source scraper | Combine data from 2 sites |
| Selenium + requests hybrid | Smart switching in one script |
| Data enrichment | Scrape detail pages for extra info |
| Final course project | Fully integrated end-to-end system |

> Open `05_Project_Final.ipynb` when you're ready!